# Testing

Test-set evaluation across all models: RMSE, MAE, R², SSIM.
Requires results saved by the individual training/baseline notebooks.

## 0 — Setup

In [1]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q rasterio geopandas h5py huggingface_hub scikit-learn scikit-image
    if not os.path.isdir('downscaling'):
        !git clone -q https://github.com/fresleven/downscaling.git
    %cd downscaling
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, os.path.abspath('../..'))


## 1 — Load results

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict
from model.dataset import DownscalingDataset

RESULTS   = 'results'
DATA_ROOT = 'data'

# test scenes & masks saved by baselines.ipynb
scenes     = np.load(f'{RESULTS}/test_scenes.npz', allow_pickle=True)
test_hr    = scenes['hr']
test_lr    = scenes['lr']
test_mask  = scenes['mask']       # cloud-free AND inside test spatial blocks
test_dmask = scenes['data_mask']
test_dates = scenes['dates'].tolist()

# load dataset for block bounding boxes (no data downloaded — just metadata)
test_ds = DownscalingDataset(root=DATA_ROOT, split='test', download=False)

# baseline predictions
bl        = np.load(f'{RESULTS}/baselines_preds.npz')
bi_test   = bl['bi_test']
bc_test   = bl['bc_test']
la_test   = bl['la_test']

# CNN predictions
cnn_d     = np.load(f'{RESULTS}/cnn_preds.npz',  allow_pickle=True)
cnn_test  = cnn_d['test_pred']

attn_d    = np.load(f'{RESULTS}/attn_preds.npz', allow_pickle=True)
attn_test = attn_d['test_pred']

print(f'Loaded {len(test_dates)} test dates, {len(test_ds.split_blocks)} test blocks.')


Loaded 308 test dates, 8 test blocks.


## 2 — Test metrics

In [3]:
from skimage.metrics import structural_similarity as ssim_fn
import pandas as pd

def compute_metrics(pred, hr, ds, mask):
    """RMSE and MAE are per (date x block), averaged — matches eval_loader in CNN notebooks.
    R² is per date (all blocks pooled). SSIM is per date on the full scene."""
    rmses, maes = [], []
    r2s, ssims  = [], []

    for date_idx in range(len(ds.dates)):
        date_p, date_t = [], []

        for block_id in ds.split_blocks:
            r0, r1, c0, c1 = ds.block_bboxes[block_id]
            blk_pred = pred[date_idx][r0:r1, c0:c1]
            blk_hr   = hr[date_idx][r0:r1, c0:c1]
            blk_mask = mask[date_idx][r0:r1, c0:c1]
            m = blk_mask & np.isfinite(blk_pred) & np.isfinite(blk_hr)
            if not m.any():
                continue
            p = blk_pred[m]; t = blk_hr[m]
            d = p - t
            rmses.append(float(np.sqrt(np.mean(d ** 2))))
            maes.append(float(np.mean(np.abs(d))))
            date_p.append(p); date_t.append(t)

        # per-date R² (all blocks pooled)
        if date_p:
            p_all = np.concatenate(date_p)
            t_all = np.concatenate(date_t)
            d = p_all - t_all
            ss_res = np.sum(d ** 2)
            ss_tot = np.sum((t_all - t_all.mean()) ** 2)
            r2s.append(float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float('nan'))

        # per-date SSIM: get per-pixel SSIM map, average only over valid test pixels.
        # Filling NaN with scene mean lets local windows be computed everywhere,
        # but the score is reported only at positions where GT is observed.
        p_img     = pred[date_idx].copy()
        gt_img    = hr[date_idx].copy()
        valid_ssim = mask[date_idx] & np.isfinite(p_img)
        if valid_ssim.any():
            fill   = float(np.nanmean(gt_img))
            p_img  = np.where(np.isfinite(p_img),  p_img,  fill)
            gt_img = np.where(np.isfinite(gt_img), gt_img, fill)
            dr     = float(np.nanmax(hr[date_idx]) - np.nanmin(hr[date_idx]))
            if dr > 0:
                ssim_map = ssim_fn(p_img, gt_img, data_range=dr, full=True)[1]
                ssims.append(float(ssim_map[valid_ssim].mean()))

    def m(v): return float(np.nanmean(v)) if v else float('nan')
    return {'RMSE': m(rmses), 'MAE': m(maes), 'R2': m(r2s), 'SSIM': m(ssims),
            'n_block_scenes': len(rmses)}

rows = []
for name, pred in [
        # ('Bicubic',       bi_test),
        ('BCSD',          bc_test),
        ('Lasso',         la_test),
        ('CNN',           cnn_test),
        ('Attention CNN', attn_test),
]:
    m = compute_metrics(pred, test_hr, test_ds, test_mask)
    rows.append({'Method': name,
                 'RMSE (°C)': round(m['RMSE'], 4),
                 'MAE (°C)':  round(m['MAE'],  4),
                 'R²':        round(m['R2'],   4),
                 'SSIM':      round(m['SSIM'], 4)})
    print(f"{name:15s}  RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}  "
          f"R²={m['R2']:.4f}  SSIM={m['SSIM']:.4f}")

df = pd.DataFrame(rows).set_index('Method')
df


BCSD             RMSE=20.3350  MAE=20.1257  R²=-40.2981  SSIM=0.0306
Lasso            RMSE=4.7909  MAE=3.8867  R²=-1.1457  SSIM=0.5060
CNN              RMSE=1.0972  MAE=0.8562  R²=0.9436  SSIM=0.4748
Attention CNN    RMSE=1.0845  MAE=0.8406  R²=0.9414  SSIM=0.4723


,RMSE (°C),MAE (°C),R²,SSIM
Method,,,,
BCSD,20.3350,20.1257,-40.2981,0.0306
Lasso,4.7909,3.8867,-1.1457,0.5060
CNN,1.0972,0.8562,0.9436,0.4748
Attention CNN,1.0845,0.8406,0.9414,0.4723
